### Weaviate Setup

In [1]:
import weaviate
from weaviate.classes.config import Configure
from weaviate.classes.config import Property, DataType
from dotenv import load_dotenv
from enum import Enum
import os
import json

In [2]:
def create_dynamic_enum(name: str, values: list[str]) -> Enum:
    unique_values = list(set(values))
    if "OTHER" not in unique_values:
        unique_values.append("OTHER")

    return Enum(name, {v.upper().replace(" ", "_"): v for v in unique_values})

In [3]:
json_path = os.path.join('..', 'data', 'unique_values.json')
if os.path.exists(json_path):
    with open(json_path, 'r') as f:
        uniques = json.load(f)
        uniques["materials"] = ['_'.join(m.split(' ')).lower() for m in uniques["materials"]]
else:
    raise Exception("Run first notebook to get products data.")


SizesEnum = create_dynamic_enum("SizesEnum", uniques.get("sizes", []))
SexEnum = create_dynamic_enum("SexEnum", uniques.get("sex", []))
MaterialEnum = create_dynamic_enum("MaterialEnum", uniques.get("materials", []))
ConstructionEnum = create_dynamic_enum("ConstructionEnum", uniques.get("constructions", []))

In [45]:
load_dotenv(dotenv_path="../.env")


def create_product_collection():
    client = weaviate.connect_to_local()

    if client.collections.exists("Products"):
        print('Collection "Products" already exists. Skipping creation.')
        client.close()
        return

    properties = [
        Property(name="name", data_type=DataType.TEXT),
        Property(name="description", data_type=DataType.TEXT),
        Property(name="color", data_type=DataType.TEXT),
        Property(name="construction", data_type=DataType.TEXT),
        Property(name="price_pln", data_type=DataType.NUMBER),
        Property(name="sizes", data_type=DataType.TEXT_ARRAY),
        Property(name="sex", data_type=DataType.TEXT),
        Property(name="image_url", data_type=DataType.TEXT),
        Property(name="embedding_string", data_type=DataType.TEXT)
    ]
    for m in MaterialEnum:
        if m.value != 'OTHER':
            properties.append(Property(name=f"{m.value}_percentage", data_type=DataType.NUMBER))

    client.collections.create(
        name="Products",
        vector_config=[Configure.Vectors.text2vec_azure_openai(
            name="embedding",
            resource_name="malgrzllm",
            deployment_id="text-embedding-ada-002",
            source_properties=["embedding_string"]
        )],
        properties=properties
    )

    client.close()

In [44]:
create_product_collection()

### Creating index

In [46]:
import pandas as pd
import os
import ast
import json

from product_dto import ProductDto, Material, Price

In [47]:
def generate_rag_string(row):
        materials_objs = []
        for key in row.keys():
            if "percentage" in key:
                name = " ".join(key.split('_')[:-1]).lower()
                materials_objs.append(
                     Material(name=name, percentage=int(float(row[key])))
                )

        price_obj = Price(value=row['price_pln'], currency="PLN")

        dto = ProductDto(
            name=row['name'],
            description=row['description'],
            color=row['color'],
            construction=row['construction'],
            materials=materials_objs,
            price=price_obj,
            sizes=row['sizes'],
            sex=row['sex'],
            image_url=row['image_url']
        )
        return dto.to_rag_string()

In [4]:
load_dotenv(dotenv_path="../.env")


def create_index_and_feed_db():
    client = weaviate.connect_to_local()
    if not client.collections.exists("Products"):
        print("Collection does not exist. Run create_product_collection() first.")
        client.close()
        return

    products_collection = client.collections.use("Products")
    if len(products_collection) > 0:
        print("Collection is not empty. Skipping data ingestion to avoid duplicates.")
        client.close()
        return

    path = os.path.join("..", "data", "products_raw.csv")
    if not os.path.exists(path):
        print(f"File {path} not found.")
        client.close()
        return

    products = pd.read_csv(path)

    objects_columns = ["sizes", "materials", "price"]
    for col in objects_columns:
        products[col] = products[col].apply(
            lambda x: ast.literal_eval(x) if isinstance(x, str) else x
        )

    def parse_materials(mats_list):
        if not isinstance(mats_list, list):
            return {}
        return {
            f"{'_'.join(m['name'].split(' ')).lower()}_percentage": float(m['percentage'])
            for m in mats_list
        }

    materials_df = products['materials'].apply(parse_materials).apply(pd.Series)
    materials_df = materials_df.fillna(0)
    products = pd.concat([products, materials_df], axis=1)

    products["embedding_string"] = products.apply(generate_rag_string, axis=1)

    print("Starting data ingestion...")
    with products_collection.batch.fixed_size(batch_size=20) as batch:
        for i, (_, row) in enumerate(products.iterrows()):
            weaviate_obj = dict(row.copy())
            weaviate_obj.pop('materials', None)
            weaviate_obj.pop('price', None)

            batch.add_object(weaviate_obj)
            if i % 20 == 0:
                print(f"Added batch up to index {i}")

    if len(products_collection.batch.failed_objects) > 0:
        print(f"Number of failed imports: {len(products_collection.batch.failed_objects)}")
        print(f"First failed object: {products_collection.batch.failed_objects[0]}")
    else:
        print("Data ingestion finished successfully.")

    client.close()

In [49]:
create_index_and_feed_db()

Starting data ingestion...
Added batch up to index 0
Added batch up to index 20
Added batch up to index 40
Added batch up to index 60
Added batch up to index 80
Added batch up to index 100
Added batch up to index 120
Added batch up to index 140
Added batch up to index 160
Added batch up to index 180
Added batch up to index 200
Added batch up to index 220
Added batch up to index 240
Added batch up to index 260
Added batch up to index 280
Added batch up to index 300
Added batch up to index 320
Added batch up to index 340
Added batch up to index 360
Added batch up to index 380
Added batch up to index 400
Added batch up to index 420
Added batch up to index 440
Added batch up to index 460
Added batch up to index 480
Added batch up to index 500
Added batch up to index 520
Added batch up to index 540
Added batch up to index 560
Added batch up to index 580
Added batch up to index 600
Added batch up to index 620
Added batch up to index 640
Added batch up to index 660
Added batch up to index 680

### Dummy 'tool'

In [4]:
from weaviate.classes.query import Filter
from google.adk.agents.llm_agent import LlmAgent

In [5]:
import typing
from pydantic import BaseModel, ConfigDict, Field

class PriceFilter(BaseModel):
    model_config = ConfigDict(extra="forbid")

    operator: typing.Literal["gt", "lt", "eq"] = Field(
        description="Operator for price filtering. 'gt' corresponds to 'greater than', 'lt' to 'less then' and 'eq' to 'equal'."
    )
    value: float


class MaterialFilter(BaseModel):
    model_config = ConfigDict(extra="forbid")

    operator: typing.Literal["gt", "lt", "eq", "exists"] = Field(
        description="Operator for material filtering. 'gt' corresponds to 'greater than' (material percetage), 'lt' to 'less then' and 'eq' to 'equal'. 'Exists' corresponds to sole material existance, without taking percetages into consideration."
    )
    key: MaterialEnum
    percentage: int | None = Field(description="Percentage value if needed, e.g. 50. If not needed, set it to null.")


class Filters(BaseModel):
    model_config = ConfigDict(extra="forbid")

    price: PriceFilter | None = Field(description="Always use PLN currency. If user gives other currency, calculate it to PLN.")

    sizes: list[SizesEnum] | None
    sex: SexEnum | None
    materials: list[MaterialFilter] | None
    construction: ConstructionEnum | None

In [6]:
def product_search(query: str, filters: Filters):
    weaviate_filters = []

    if filters.sex and filters.sex.value != "OTHER":
        weaviate_filters.append(Filter.by_property("sex").equal(filters.sex.value))

    if filters.construction and filters.construction.value != "OTHER":
        weaviate_filters.append(Filter.by_property("construction").equal(filters.construction.value))

    if filters.sizes:
        size_values = [s.value for s in filters.sizes if s.value != "OTHER"]
        if size_values:
            weaviate_filters.append(Filter.by_property("sizes").contains_any(size_values))

    if filters.price:
        if filters.price.operator == "gt":
            weaviate_filters.append(Filter.by_property("price_pln").greater_than(filters.price.value))
        elif filters.price.operator == "lt":
            weaviate_filters.append(Filter.by_property("price_pln").less_than(filters.price.value))

    if filters.materials:
        for mat in filters.materials:
            if mat.key.value == "OTHER":
                continue

            if mat.operator == "exists":
                weaviate_filters.append(
                    Filter.by_property(f"{mat.key.value}_percentage").greater_than(0)
                )

            elif mat.operator == "eq" and mat.percentage is not None:
                weaviate_filters.append(
                    Filter.by_property(f"{mat.key.value}_percentage").equal(mat.percentage)
                )

            elif mat.operator == "gt" and mat.percentage is not None:
                weaviate_filters.append(
                    Filter.by_property(f"{mat.key.value}_percentage").greater_than(mat.percentage)
                )

            elif mat.operator == "lt" and mat.percentage is not None:
                weaviate_filters.append(
                    Filter.by_property(f"{mat.key.value}_percentage").less_than(mat.percentage)
                )

            else:
                weaviate_filters.append(
                    Filter.by_property(f"{mat.key.value}_percentage").greater_than(0)
                )

    combined_filter = None
    if weaviate_filters:
        combined_filter = weaviate_filters[0]
        for f in weaviate_filters[1:]:
            combined_filter = combined_filter & f

    with weaviate.connect_to_local() as client:
        collection = client.collections.use("Products")

        response = collection.query.hybrid(
            query=query,
            filters=combined_filter,
            limit=5
        )

        if not response.objects:
            message = "No results after search with filters. Provided results were obtained without the filters."
            response = collection.query.hybrid(query=query, limit=3)
        else:
            message = "Success in obtaining requested products with given query and filters."

        return {"message": message, "search_results": [obj.properties for obj in response.objects]}

In [7]:
from google.genai import types
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.models.lite_llm import LiteLlm


async def perform_prompt(userId: str, user_input: str, model):
    session_service = InMemorySessionService()
    session = await session_service.create_session(
        app_name="llm",
        user_id=userId,
    )
    runner = Runner(
        agent=model,
        app_name="llm",
        session_service=session_service,
    )

    input_message = types.Content(role="user", parts=[types.Part(text=user_input)])

    response = "Something went wrong and I can't answer your question :<"

    for event in runner.run(user_id=userId, session_id=session.id, new_message=input_message):
        if event.is_final_response():
            if event.content and event.content.parts:
                response = event.content.parts[0].text
            elif event.actions and event.actions.escalate:
                response = f"Agent escalated: {event.error_message or 'No specific message.'}"
            break

    return response

In [8]:
async def run_gen_ai_query(user_query: str):
    prompt = f"""
    You are a search assistant for a clothing store.
    Analyze the user query and extract filters.
    If a specific attribute is not mentioned, leave it null.

    User Query: "{user_query}"
    """

    model = LlmAgent(
        model=LiteLlm(model='azure/gpt-4o'),
        name="clothing_query_creator",
        description="Generates search queries and filters for user's query",
        instruction=prompt,
        output_schema=Filters
    )
    # response = await perform_prompt("123", user_query, model)

    try:
        # filters_json = json.loads(response)
        # filters_obj = Filters(**filters_json)
        filters_obj = Filters(
            price=None, sizes=['M'], sex='Women', materials=[MaterialFilter(operator='exists', key='polyester', percentage=None)], construction=None
        )
        results = product_search(user_query, filters_obj)
        return results

    except Exception as e:
        print(f"Parsing error: {e}")
        return []

In [9]:
import logging
import sys

logging.basicConfig(
    level=logging.DEBUG,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    stream=sys.stdout
)

logging.getLogger("httpcore").setLevel(logging.WARNING)
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("aiosqlite").setLevel(logging.INFO)

In [13]:
response = await run_gen_ai_query("I need a cheap cotton t-shirt size M for woman")
print(json.dumps(response, indent=4))

{
    "message": "Success in obtaining requested products with given query and filters.",
    "search_results": [
        {
            "polyester_percentage": 100.0,
            "cashmere_percentage": 0.0,
            "sex": "Women",
            "other_fibres_percentage": 0.0,
            "acrylic_percentage": 0.0,
            "name": "Cotton T-shirt",
            "cotton_percentage": 0.0,
            "rubber_percentage": 0.0,
            "lyocell_percentage": 0.0,
            "price_pln": 63.0,
            "image_url": "https://image.hm.com/assets/hm/89/35/893510cf78c3aff0d956f5d449639ac7732ed657.jpg",
            "construction": "Knitted",
            "description": "T-shirt in dense cotton jersey with a round, rib-trimmed neckline.",
            "paper_percentage": 0.0,
            "elastane_percentage": 0.0,
            "elastodiene_percentage": 0.0,
            "mohair_percentage": 0.0,
            "viscose_percentage": 0.0,
            "color": "Burgundy",
            "modacryli